In [24]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
import json

In [25]:
def load_data(file_path):
    if not file_path:
        raise FileNotFoundError("File path is invalid.")
    data = pd.read_csv(file_path)
    if 'Tags' not in data.columns or 'Province' not in data.columns:
        raise ValueError("Dataset must contain 'Tags' and 'Province' columns.")
    data['Tags'] = data['Tags'].apply(lambda x: x.split() if isinstance(x, str) else [])
    return data

In [26]:
def load_categorized_tags(json_path):
    with open(json_path, 'r') as f:
        categorized_tags = json.load(f)
    return categorized_tags

In [27]:
def determine_category(tags, categorized_tags):
    for category, category_tags in categorized_tags.items():
        if any(tag in category_tags for tag in tags):
            return category
    return 'Uncategorized'

In [28]:
def create_feature_matrix(data, categorized_tags):
    # Assign categories to attractions based on their tags
    data['Category'] = data['Tags'].apply(lambda x: determine_category(x, categorized_tags))
    
    # Encode provinces and categories
    province_encoder = LabelEncoder()
    category_encoder = LabelEncoder()
    
    data['Encoded_Province'] = province_encoder.fit_transform(data['Province'])
    data['Encoded_Category'] = category_encoder.fit_transform(data['Category'])
    
    # Encode tags using MultiLabelBinarizer
    mlb = MultiLabelBinarizer()
    tag_features = pd.DataFrame(mlb.fit_transform(data['Tags']), columns=mlb.classes_, index=data.index)
    
    # Combine all features into a single feature matrix
    feature_matrix = pd.concat([data[['Encoded_Province', 'Encoded_Category']], tag_features], axis=1)
    
    return feature_matrix, mlb, province_encoder, category_encoder

In [29]:
def create_user_profile(province, category, tags, feature_matrix, mlb, province_encoder, category_encoder):
    user_profile = pd.DataFrame(0, index=[0], columns=feature_matrix.columns)
    
    # Encode province and category into the user profile
    encoded_province = province_encoder.transform([province])[0]
    encoded_category = category_encoder.transform([category])[0]
    
    user_profile['Encoded_Province'] = encoded_province
    user_profile['Encoded_Category'] = encoded_category
    
    # Encode tags into the user profile
    for tag in tags:
        if tag in user_profile.columns:
            user_profile[tag] = 1
    
    return user_profile

In [30]:
def recommend_attractions(data, feature_matrix, user_profile, province, top_n=5):
    # Filter attractions by selected province
    filtered_data = data[data['Province'].str.strip().str.lower() == province.strip().lower()]
    
    if filtered_data.empty:
        print(f"No attractions found in the selected province: {province}")
        return pd.DataFrame()
    
    # Filter feature matrix based on filtered data indices
    filtered_feature_matrix = feature_matrix.loc[filtered_data.index]
    
    # Calculate cosine similarity between user profile and filtered attractions
    similarities = cosine_similarity(user_profile, filtered_feature_matrix)
    
    # Get top N recommendations based on similarity scores
    top_indices = np.argsort(similarities[0])[-top_n:][::-1]
    
    recommendations = filtered_data.iloc[top_indices][['ID', 'Name', 'Province', 'Tags']]
    
    return recommendations

In [31]:
def main():
    try:
        # Load dataset and categorized tags
        data_file_path = 'Output/PreparedData.csv'
        categorized_tags_path = '../Data/FinalDataset/CategorizedTags.json'
        
        data = load_data(data_file_path)
        categorized_tags = load_categorized_tags(categorized_tags_path)
        
        # Create feature matrix and encoders
        feature_matrix, mlb, province_encoder, category_encoder = create_feature_matrix(data, categorized_tags)
        
        # Get user preferences (replace with actual input or interactive prompts)
        selected_province = input("Enter your preferred province: ").strip()
        selected_category = input("Enter your preferred category: ").strip()
        
        print("\nAvailable Tags:")
        print(", ".join(mlb.classes_))
        
        selected_tags_input = input("Enter your preferred tags (comma-separated): ").strip()
        selected_tags = [tag.strip() for tag in selected_tags_input.split(',')]
        
        # Create user profile based on preferences
        user_profile = create_user_profile(selected_province, selected_category,
                                           selected_tags, feature_matrix,
                                           mlb, province_encoder,
                                           category_encoder)
        
        # Generate recommendations
        recommendations = recommend_attractions(data, feature_matrix,
                                                 user_profile,
                                                 selected_province,
                                                 top_n=5)
        
        if recommendations.empty:
            print("\nNo recommendations found.")
        else:
            print("\nRecommended Attractions:")
            print(recommendations.to_string(index=False))
    
    except Exception as e:
        print(f"Error: {e}")

# Cell 9: Run Main Function (Uncomment to Execute in Jupyter Notebook)
if __name__ == "__main__":
    main()


Available Tags:
accommodationdining, activity, adventure, adventurenatural, adventuretourist, agencycultural, agencytourism, agencytourismcultural, agriculture, agriculturerural, agriculturetourism, amusement, amusementpark, amusementparkhikingnatural, amusementparknatural, amusementparknature, amusementparkrecreational, and, archaeological, area, areabusy, areanatural, areatourist, art, artarts, artcultural, artgallery, artgallerycultural, artgalleryculturalmuseum, artgalleryfurniture, artgalleryjewelrystorefurniturestorecultural, artgallerymuseumcultural, asset, attraction, attractionbiodiversity, attractionbuddhist, attractioncultural, attractioneconomic, attractionhiking, attractionhikingnatural, attractionhistoric, attractionhistorical, attractioninfrastructure, attractionnatural, attractionnature, attractionoutdoor, attractionrecreation, attractionrecreational, attractionrecreationnatural, attractionreligious, attractionresortoutdoor, attractionrural, attractionsafari, attractio

In [36]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
import json

# Cell 2: Load Dataset
def load_data(file_path):
    if not file_path:
        raise FileNotFoundError("File path is invalid.")
    data = pd.read_csv(file_path)
    if 'Tags' not in data.columns or 'Province' not in data.columns:
        raise ValueError("Dataset must contain 'Tags' and 'Province' columns.")
    data['Tags'] = data['Tags'].apply(lambda x: x.split() if isinstance(x, str) else [])
    return data

# Cell 3: Load Categorized Tags
def load_categorized_tags(json_path):
    with open(json_path, 'r') as f:
        categorized_tags = json.load(f)
    return categorized_tags

# Cell 4: Determine Category Based on Tags
def determine_category(tags, categorized_tags):
    for category, category_tags in categorized_tags.items():
        if any(tag in category_tags for tag in tags):
            return category
    return 'Uncategorized'

# Cell 5: Create Feature Matrix
def create_feature_matrix(data, categorized_tags):
    # Assign categories to attractions based on their tags
    data['Category'] = data['Tags'].apply(lambda x: determine_category(x, categorized_tags))
    
    # Encode provinces and categories
    province_encoder = LabelEncoder()
    category_encoder = LabelEncoder()
    
    data['Encoded_Province'] = province_encoder.fit_transform(data['Province'])
    data['Encoded_Category'] = category_encoder.fit_transform(data['Category'])
    
    # Encode tags using MultiLabelBinarizer
    mlb = MultiLabelBinarizer()
    tag_features = pd.DataFrame(mlb.fit_transform(data['Tags']), columns=mlb.classes_, index=data.index)
    
    # Combine all features into a single feature matrix
    feature_matrix = pd.concat([data[['Encoded_Province', 'Encoded_Category']], tag_features], axis=1)
    
    return feature_matrix, mlb, province_encoder, category_encoder

# Cell 6: Create User Profile
def create_user_profile(province, category, tags, feature_matrix, mlb, province_encoder, category_encoder):
    user_profile = pd.DataFrame(0, index=[0], columns=feature_matrix.columns)
    
    # Encode province and category into the user profile
    encoded_province = province_encoder.transform([province])[0]
    encoded_category = category_encoder.transform([category])[0]
    
    user_profile['Encoded_Province'] = encoded_province
    user_profile['Encoded_Category'] = encoded_category
    
    # Encode tags into the user profile
    for tag in tags:
        if tag in user_profile.columns:
            user_profile[tag] = 1
    
    return user_profile

# Cell 7: Recommend Attractions
def recommend_attractions(data, feature_matrix, user_profile, province, top_n=5):
    # Filter attractions by selected province
    filtered_data = data[data['Province'].str.strip().str.lower() == province.strip().lower()]
    
    if filtered_data.empty:
        print(f"No attractions found in the selected province: {province}")
        return pd.DataFrame()
    
    # Filter feature matrix based on filtered data indices
    filtered_feature_matrix = feature_matrix.loc[filtered_data.index]
    
    # Calculate cosine similarity between user profile and filtered attractions
    similarities = cosine_similarity(user_profile, filtered_feature_matrix)
    
    # Get top N recommendations based on similarity scores
    top_indices = np.argsort(similarities[0])[-top_n:][::-1]
    
    recommendations = filtered_data.iloc[top_indices][['ID', 'Name', 'Province', 'Tags']]
    
    return recommendations

# Cell 8: Display Options for User Input
def display_options(options_list, prompt):
    print(prompt)
    for i, option in enumerate(options_list, start=1):
        print(f"{i}. {option}")
    
    choice_index = int(input("\nEnter your choice (number): ")) - 1
    
    if choice_index < 0 or choice_index >= len(options_list):
        raise ValueError("Invalid choice. Please select a valid option.")
    
    return options_list[choice_index]

# Cell 9: Main Function to Run the Recommendation System with Options
def main():
    try:
        # Load dataset and categorized tags
        data_file_path = 'Output/PreparedData.csv'
        categorized_tags_path = '../Data/FinalDataset/CategorizedTags.json'
        
        data = load_data(data_file_path)
        categorized_tags = load_categorized_tags(categorized_tags_path)
        
        # Create feature matrix and encoders
        feature_matrix, mlb, province_encoder, category_encoder = create_feature_matrix(data, categorized_tags)
        
        # Display available provinces for selection
        provinces = list(data['Province'].unique())
        selected_province = display_options(provinces, "Select a Province:")
        
        # Display available categories for selection
        categories = list(categorized_tags.keys())
        selected_category = display_options(categories, "Select a Category:")
        
        # Display available tags for the selected category and allow multiple selections
        available_tags = categorized_tags[selected_category]
        print("\nAvailable Tags:")
        for i, tag in enumerate(available_tags, start=1):
            print(f"{i}. {tag}")
        
        selected_tag_indices = input("\nEnter your preferred tags (comma-separated numbers): ").split(',')
        selected_tags = [available_tags[int(index.strip()) - 1] for index in selected_tag_indices]
        
        # Create user profile based on preferences
        user_profile = create_user_profile(selected_province, selected_category,
                                           selected_tags, feature_matrix,
                                           mlb, province_encoder,
                                           category_encoder)
        
        # Generate recommendations
        recommendations = recommend_attractions(data, feature_matrix,
                                                 user_profile,
                                                 selected_province,
                                                 top_n=5)
        
        if recommendations.empty:
            print("\nNo recommendations found.")
        else:
            print("\nRecommended Attractions:")
            print(recommendations.to_string(index=False))
    
    except Exception as e:
        print(f"Error: {e}")

# Cell 10: Run Main Function (Uncomment to Execute in Jupyter Notebook)
if __name__ == "__main__":
    main()


Select a Province:
1. Bagmati Province
2. Lumbini Province
3. Koshi Province
4. Madhesh Province
5. Sudurpashchim Province
6. Karnali Province
7. Gandaki Province
Select a Category:
1. Adventure
2. Nature
3. Cultural
4. Historical
5. Recreational
6. Religious
7. Miscellaneous

Available Tags:
1. adventure
2. camping
3. canyoning
4. eco-adventure
5. hike
6. hiking
7. hiking destination
8. hiking spot
9. hiking trails
10. hiking trails.
11. kayaking
12. mountaineering expeditions
13. mundum trail
14. outdoor adventure
15. rafting
16. stargazing
17. trails
18. trek
19. trekking
20. trekking agency
21. trekking destination
22. trekking gateway
23. trekking routes
24. trekking service
25. birdwatching
26. birdwatching area
27. birdwatching spot
28. boating
29. cave exploration
30. cycling
31. hiking
32. mountaineering expeditions
33. safari destination
34. suspended bridge
35. suspension bridge

Recommended Attractions:
  ID                         Name       Province                       